# Gov24 캡차 이미지 수집

`https://www.gov.kr/mw/captcha` API를 호출하여 캡차 이미지를 다운로드합니다.

In [3]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO

In [7]:
# 저장 경로 설정
base_path = Path("captcha_data/gov24/1/images/draft")
base_path.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {base_path.absolute()}")

저장 경로: /home/hyper/project/hyper-captcha-resolver/captcha_data/gov24/1/images/draft


In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

captcha_id = 'gov24'
rev = 1
backend = 'keras'

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
train_data.rev = rev
image_width: int = 200
image_height: int = 50
engine.batch_predict_model(model=model)
model_path = train_data.get_model_path()
image_path = train_data.choice_pred_image()
pred, confidence = engine.predict(model=model, image_path=image_path)
print("image_path : ", image_path)
print("pred : ", pred)
print("confidence : ", f'{confidence:.4f}')
print("Done!")


In [8]:
def download_captcha(save_path: Path, index: int, total: int) -> bool:
    """
    캡차 이미지를 다운로드하고 저장합니다.
    
    Args:
        save_path: 저장할 경로
        index: 현재 인덱스
        total: 전체 개수
    
    Returns:
        성공 여부
    """
    url = "https://www.gov.kr/mw/captcha"
    
    try:
        # API 호출
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # 타임스탬프 기반 파일명 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = save_path / filename
        
        # 이미지로 변환 후 PNG로 저장
        image = Image.open(BytesIO(response.content))
        image.save(filepath, format="PNG")
        
        print(f"[{index + 1}/{total}] 저장 완료: {filename}")
        return True
        
    except Exception as e:
        print(f"[{index + 1}/{total}] 오류 발생: {e}")
        return False

In [9]:
# 다운로드 설정
TARGET_COUNT = 550  # 다운로드할 이미지 개수
DELAY = 0.25         # 요청 간 대기 시간 (초)

print(f"캡차 이미지 {TARGET_COUNT}개 다운로드 시작...")
print(f"요청 간격: {DELAY}초")
print("-" * 50)

캡차 이미지 550개 다운로드 시작...
요청 간격: 0.25초
--------------------------------------------------


In [10]:
# 이미지 다운로드 실행
success_count = 0
fail_count = 0

for i in range(TARGET_COUNT):
    if download_captcha(base_path, i, TARGET_COUNT):
        success_count += 1
    else:
        fail_count += 1
    
    # 마지막 요청이 아니면 대기
    if i < TARGET_COUNT - 1:
        time.sleep(DELAY)

print("-" * 50)
print(f"\n다운로드 완료!")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 위치: {base_path.absolute()}")

[1/550] 저장 완료: 20251104_145031_240482.png
[2/550] 저장 완료: 20251104_145031_676910.png
[2/550] 저장 완료: 20251104_145031_676910.png
[3/550] 저장 완료: 20251104_145032_105412.png
[3/550] 저장 완료: 20251104_145032_105412.png
[4/550] 저장 완료: 20251104_145032_507293.png
[4/550] 저장 완료: 20251104_145032_507293.png
[5/550] 저장 완료: 20251104_145032_917443.png
[5/550] 저장 완료: 20251104_145032_917443.png
[6/550] 저장 완료: 20251104_145033_271444.png
[6/550] 저장 완료: 20251104_145033_271444.png
[7/550] 저장 완료: 20251104_145033_638276.png
[7/550] 저장 완료: 20251104_145033_638276.png
[8/550] 저장 완료: 20251104_145034_029851.png
[8/550] 저장 완료: 20251104_145034_029851.png
[9/550] 저장 완료: 20251104_145034_445333.png
[9/550] 저장 완료: 20251104_145034_445333.png
[10/550] 저장 완료: 20251104_145034_836397.png
[10/550] 저장 완료: 20251104_145034_836397.png
[11/550] 저장 완료: 20251104_145035_586953.png
[11/550] 저장 완료: 20251104_145035_586953.png
[12/550] 저장 완료: 20251104_145036_772675.png
[12/550] 저장 완료: 20251104_145036_772675.png
[13/550] 저장 완료: 20251104_145

## 캡차 이미지 인식 및 파일명 변경

학습된 모델을 사용하여 draft 폴더의 이미지를 인식하고, 예측된 레이블로 파일명을 변경합니다.

In [11]:
import os
from pathlib import Path
import tensorflow as tf
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

draft_image_dir = "captcha_data/gov24/1/images/draft"
draft_image_files = sorted([str(p) for p in Path(draft_image_dir).glob("*.png") if len(p.name) > 10])
print(f"Draft 이미지 개수: {len(draft_image_files)}")

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height
keras_model: KerasModel = model
matched = 0
pred_img_path_list = keras_model.train_data.get_data_files(train=False)
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

for idx, img_path in enumerate(draft_image_files):
    pred, confidence = engine.predict(model=model, image_path=img_path, verbose=0)
    # rename draft image with predicted text
    new_image_path = os.path.join(draft_image_dir, pred + ".png")
    if(os.path.exists(new_image_path)):
        continue
    Path(img_path).rename(new_image_path)
    print(f"[{idx + 1}/{len(draft_image_files)}] image_path : {img_path}")
    print(f"pred : {pred}")
    print(f"confidence : {confidence:.4f}")
    print("new_image_path : ", new_image_path)
    print("Done!")


2025-11-04 15:09:51.842638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2025-11-04 15:09:51.842638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Draft 이미지 개수: 549


2025-11-04 15:09:51.842638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Draft 이미지 개수: 549


I0000 00:00:1762236595.406083  392070 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3449 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080, pci bus id: 0000:01:00.0, compute capability: 7.5
2025-11-04 15:09:57.059439: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2025-11-04 15:09:57.059439: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


2025-11-04 15:09:51.842638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Draft 이미지 개수: 549


I0000 00:00:1762236595.406083  392070 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3449 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080, pci bus id: 0000:01:00.0, compute capability: 7.5
2025-11-04 15:09:57.059439: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2025-11-04 15:09:57.059439: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


[1/549] image_path : captcha_data/gov24/1/images/draft/20251104_145031_240482.png
pred : 098798
confidence : 0.9958
new_image_path :  captcha_data/gov24/1/images/draft/098798.png
Done!
[2/549] image_path : captcha_data/gov24/1/images/draft/20251104_145031_676910.png
pred : 981018
confidence : 0.9851
new_image_path :  captcha_data/gov24/1/images/draft/981018.png
Done!
[3/549] image_path : captcha_data/gov24/1/images/draft/20251104_145032_105412.png
pred : 732397
confidence : 0.9724
new_image_path :  captcha_data/gov24/1/images/draft/732397.png
Done!
[4/549] image_path : captcha_data/gov24/1/images/draft/20251104_145032_507293.png
pred : 419540
confidence : 0.9622
new_image_path :  captcha_data/gov24/1/images/draft/419540.png
Done!
[5/549] image_path : captcha_data/gov24/1/images/draft/20251104_145032_917443.png
pred : 079222
confidence : 0.9875
new_image_path :  captcha_data/gov24/1/images/draft/079222.png
Done!
[6/549] image_path : captcha_data/gov24/1/images/draft/20251104_145033_2714

In [13]:
# 이미지 리사이즈 및 크롭 (오른쪽 여백 제거)
from pathlib import Path
from PIL import Image

# 경로 설정
source_dir = Path("captcha_data/gov24/0/images/draft-thin")
target_dir = Path("captcha_data/gov24/0/images/resize")
target_dir.mkdir(parents=True, exist_ok=True)

# 이미지 파일 목록
image_files = list(source_dir.glob("*.png"))
print(f"처리할 이미지 개수: {len(image_files)}")

# 각 이미지 처리
success_count = 0
for idx, img_path in enumerate(image_files, 1):
    try:
        # 이미지 열기 (221 * 81)
        img = Image.open(img_path)

        # 위쪽, 왼쪽 1픽셀 크롭
        img_cropped = img.crop((1, 1, 221, 81))
        
        # 225x50으로 리사이즈
        img_resized = img_cropped.resize((200, 50), Image.Resampling.LANCZOS)
        
        # 저장 (파일명 유지)
        target_path = target_dir / img_path.name
        img_resized.save(target_path, format="PNG")
        
        success_count += 1
        
        if idx % 50 == 0 or idx == len(image_files):
            print(f"[{idx}/{len(image_files)}] 처리 완료")
            
    except Exception as e:
        print(f"[{idx}/{len(image_files)}] 오류 발생 ({img_path.name}): {e}")

print(f"\n작업 완료!")
print(f"성공: {success_count}/{len(image_files)}")
print(f"저장 위치: {target_dir.absolute()}")

처리할 이미지 개수: 910
[50/910] 처리 완료
[100/910] 처리 완료
[150/910] 처리 완료
[200/910] 처리 완료
[250/910] 처리 완료
[300/910] 처리 완료
[350/910] 처리 완료
[400/910] 처리 완료
[450/910] 처리 완료
[500/910] 처리 완료
[550/910] 처리 완료
[600/910] 처리 완료
[650/910] 처리 완료
[700/910] 처리 완료
[750/910] 처리 완료
[800/910] 처리 완료
[850/910] 처리 완료
[900/910] 처리 완료
[910/910] 처리 완료

작업 완료!
성공: 910/910
저장 위치: /home/hyper/project/hyper-captcha-resolver/captcha_data/gov24/0/images/resize


In [15]:
# target_dir = Path("captcha_data/gov24/0/images/resize")
import glob


image_list = glob.glob(os.path.join(target_dir, "*.png"))
print(f"리사이즈된 이미지 개수: {len(image_list)}")

리사이즈된 이미지 개수: 1906


In [ ]:
import os, glob, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import keras
import tensorflow as tf

from pathlib import Path
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

target_dir = Path("captcha_data/gov24/1/images/labeled")
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

start = time.time()

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height

keras_model: KerasModel = model
matched = 0

pred_img_path_list = sorted(glob.glob(os.path.join(target_dir, "*.png")))
pred_labels = [os.path.basename(p).split(".")[0] for p in pred_img_path_list]
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list, pred_labels))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

# Batch prediction
all_preds = []
all_labels = []

for batch in pred_dataset:
    images = batch["image"]
    labels = batch["label"]
    
    # Predict batch
    pred_vals = keras_model.predict_model.predict(images, verbose=0)
    preds = keras_model.decode_batch_predictions(pred_vals)
    
    # Decode original labels
    for label in labels:
        label_text = tf.strings.reduce_join(
            keras_model.num_to_char(label + 1)
        ).numpy().decode("utf-8")
        all_labels.append(label_text)
    
    all_preds.extend(preds)

# Compare predictions with original labels
for idx, (ori, pred) in enumerate(zip(all_labels, all_preds)):
    img_path = pred_img_path_list[idx]
    msg = ""
    if ori == pred:
        train_img_path = train_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, train_img_path, overwrite=True)
        matched += 1
    else:
        # 불일치 파일은 pred_dir로 복사
        pred_img_path = pred_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, pred_img_path, overwrite=True)
        msg = " Not matched!"
    
    # Calculate confidence for display (optional)
    print(f"ori: {ori}, pred: {pred}{msg}")

end = time.time()
total = len(pred_img_path_list)
accuracy = matched / total * 100 if total > 0 else 0

print(f"Matched: {matched}, Total: {total}, Accuracy: {accuracy:.2f}%")
print(f"pred time: {end - start:.2f} sec") 


# engine.batch_predict_model(model=model)
# model_path = train_data.get_model_path()
# image_path = train_data.choice_pred_image()
# pred, confidence = engine.predict(model=model, image_path=image_path)
# print("image_path : ", image_path)
# print("pred : ", pred)
# print("confidence : ", f'{confidence:.4f}')
# print("Done!")


In [1]:
# captcha_data/gov24/1/images/pred 폴더의 이미지 크기 검사
from pathlib import Path
from PIL import Image

# 경로 설정
pred_dir = Path("captcha_data/gov24/1/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 크기가 다른 이미지 리스트
    wrong_size_images = []
    target_width = 200
    target_height = 50
    
    for img_path in image_files:
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            if width != target_width or height != target_height:
                wrong_size_images.append({
                    'name': img_path.name,
                    'size': f"{width}x{height}"
                })
                
        except Exception as e:
            print(f"⚠️  오류 ({img_path.name}): {e}")
    
    # 결과 출력
    if wrong_size_images:
        print(f"\n❌ 크기가 {target_width}x{target_height}이 아닌 이미지 ({len(wrong_size_images)}개):\n")
        for idx, img_info in enumerate(wrong_size_images, 1):
            print(f"  {idx:3d}. {img_info['name']:30s} -> {img_info['size']}")
    else:
        print(f"\n✅ 모든 이미지가 {target_width}x{target_height} 크기입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 ({target_width}x{target_height}): {len(image_files) - len(wrong_size_images)}개")
    print(f"비정상: {len(wrong_size_images)}개")

검사할 이미지 개수: 1906

✅ 모든 이미지가 200x50 크기입니다.

검사 완료!
전체: 1906개
정상 (200x50): 1906개
비정상: 0개


In [3]:
# captcha_data/gov24/1/images/pred 폴더의 파일명 길이 검사 (확장자 포함 10자리가 아닌 것)
from pathlib import Path

# 경로 설정
pred_dir = Path("captcha_data/gov24/0/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 파일명 길이가 10자리(확장자 포함)가 아닌 이미지 리스트
    wrong_name_images = []
    target_length = 10  # 예: "abc12.png" = 9자 (레이블 5자 + ".png" 4자)
    
    for img_path in image_files:
        filename = img_path.name
        name_length = len(filename)
        
        if name_length != target_length:
            wrong_name_images.append({
                'name': filename,
                'length': name_length,
                'label_length': len(img_path.stem)  # 확장자 제외한 레이블 길이
            })
    
    # 결과 출력
    if wrong_name_images:
        print(f"\n❌ 파일명 길이가 {target_length}자가 아닌 이미지 ({len(wrong_name_images)}개):\n")
        for idx, img_info in enumerate(wrong_name_images, 1):
            print(f"  {idx:3d}. {img_info['name']:40s} (길이: {img_info['length']:2d}, 레이블: {img_info['label_length']:2d}자)")
    else:
        print(f"\n✅ 모든 이미지 파일명이 {target_length}자입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 (파일명 {target_length}자): {len(image_files) - len(wrong_name_images)}개")
    print(f"비정상: {len(wrong_name_images)}개")

검사할 이미지 개수: 1906

❌ 파일명 길이가 10자가 아닌 이미지 (1개):

    1. 28942.png                                (길이:  9, 레이블:  5자)

검사 완료!
전체: 1906개
정상 (파일명 10자): 1905개
비정상: 1개


In [ ]:
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32
model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
